# this is running the estimator for the pz challenge taskset 1

In [1]:
import sys

from packaging import version
import sklearn
from sklearn.model_selection import KFold, train_test_split

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import pandas as pd
import h5py
import numpy as np

import json
import os

from matplotlib import colors
import pickle
from scipy.stats import sigmaclip
import matplotlib.gridspec as gridspec
from tensorflow.keras import layers, models, callbacks

import cnnpz_utils as cnnpz

2026-09-14 09:24:36.949575: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-14 09:24:36.949602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-14 09:24:36.950897: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-14 09:24:36.958867: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-14 09:24:39.543274: W tensorflow/compiler/tf2

# predefined functions

## Load training and test data, filter curves

In [2]:
saveroot = "/pscratch/sd/j/jaimerz/pz_datachllenge_data/"
sims = ["cardinal","flagship"]
Yrs = [1, 10]
test_dataset_taskset1 = {}
for sim in sims:
    test_dataset_taskset1[sim] = {}
    for Yr in Yrs:
        fname = saveroot + f"pz_challenge_taskset_1_{sim}_test_{Yr}yr.hdf5"
        with h5py.File(fname, 'r') as f:
            data = {key: f[key][:] for key in f.keys()}
        test_dataset_taskset1[sim][Yr] = pd.DataFrame(data)

train_dataset_taskset1 = {}
for sim in sims:
    train_dataset_taskset1[sim] = {}
    for Yr in Yrs:
        fname = saveroot + f"pz_challenge_taskset_1_{sim}_training_{Yr}yr.hdf5"
        with h5py.File(fname, 'r') as f:
            data = {key: f[key][:] for key in f.keys()}
        train_dataset_taskset1[sim][Yr] = pd.DataFrame(data)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/pscratch/sd/j/jaimerz/pz_datachllenge_data/pz_challenge_taskset_1_cardinal_test_1yr.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
data.keys()

In [ ]:
# get the LSST and roman filter curves:
filter_root = "/global/homes/j/jaimerz/UCL/rail_base/src/rail/examples_data/estimation_data/data/FILTER/"

wave = {
    "Y":106,
    "J":129,
    "H":158,
}
lsst_filter_curves = {}
roman_filter_curves = {}
for b in "ugrizy":
  lsst_filter_curves[b] = np.loadtxt(filter_root + f'DC2LSST_{b}.res')

for b in "YJH":
  roman_filter_curves[b] = np.loadtxt(filter_root + f'roman_{b}{wave[b]}.res')

# define the wavelength grid
# to begin with, use blocks rather than filter curves
lambda_min = lsst_filter_curves['u'][:,0].min()
lambda_max = roman_filter_curves['H'][:,0].max()
print(lambda_min,lambda_max)

lambda_array = np.linspace(lambda_min,lambda_max,31)
# now let's conver filter curves to blocks: here let's also ignore the big Y band as it overlaps with the y band

### Transform data, make X_train, Y_train, and test dataset, variants of this dataset

In [ ]:
bin_edges = {}
for b in "ugrizyJH":
  if b not in "JH":
    bin_edges[b] = cnnpz.get_bin_edges(lsst_filter_curves[b][:,0])
  else:
    bin_edges[b] = cnnpz.get_bin_edges(roman_filter_curves[b][:,0])
new_edges = lambda_array

rebinned_filters={}
for b in "ugrizyJH":
  if b not in "JH":
    counts = lsst_filter_curves[b][:,1]
  else:
    counts = roman_filter_curves[b][:,1]
  rebinned_filters[b] = cnnpz.rebin_filter(bin_edges[b], counts, new_edges)

lambda_array_cen = (new_edges[1:] + new_edges[:-1])/2
dlambda = new_edges[1] - new_edges[0]
#for b in "ugrizyJH":
#  plt.plot(lambda_array_cen, rebinned_filters[b],'.-')

# further convert this to blocks:
# let's take 1 when the current filter value is > next filter
filter_blocks={}
bands = "ugrizyJH"
for i, b in enumerate("ugrizyJ"):
  if i>0:
    filter_blocks[b] = (rebinned_filters[b] >= rebinned_filters[bands[i+1]]) & (rebinned_filters[b] > rebinned_filters[bands[i-1]])
  else:
    filter_blocks[b] = (rebinned_filters[b] > rebinned_filters[bands[i+1]])
filter_blocks['H'] = (rebinned_filters['H'] > rebinned_filters['J']) & (rebinned_filters['H'] > rebinned_filters['y'])

# Cardinal Y1

In [ ]:
# let's start with Carindal Y1 T1:
training_data = train_dataset_taskset1['cardinal'][1]
test_data = test_dataset_taskset1['cardinal'][1]

# now split training and validation set:
X, Y = cnnpz.transform_data_to_XY(training_data, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test, _ = cnnpz.transform_data_to_XY(test_data, lambda_array_cen, filter_blocks, apply_stretch = False, missingY=True)

In [ ]:
cnnpz.visualize_the_data(X, Y, lambda_array_cen, filter_blocks, title="Y1 training example")

# training ensemble model

In [ ]:
trained_models, histories = cnnpz.train_ensembles(build_model, X, Y)

In [ ]:
cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.1))

In [ ]:
# Final prediction on test set
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test)

In [ ]:
y_pred_ensemble2, y_pred_STD2 = cnnpz.ensemble_predict(trained_models, X)

In [ ]:
# save the trained model:
root = "/pscratch/sd/j/jaimerz/cnnpz/PZDC/taskset1/Cardinal_Y1/models"
save_dir = root + "/ensemble_CNN_6layers"
cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
cc = plt.hist(y_pred_STD.flatten(), bins=20)
plt.yscale("log")

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
fname = save_dir + "/stats.pkl"
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y, y_pred_ensemble2.flatten(), training_data['mag_i_lsst'], save=True, saveroot=fname, 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y, y_pred_ensemble2.flatten(), 
           redshift_bins, imag_bins, training_data['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# also plot the STD as a function of redshift and i-band magnitude:
fig,axarr=plt.subplots(1,3,figsize=[13,4])
plt.sca(axarr[0])
ind = y_pred_STD2.flatten() >= 0.05
plt.scatter(Y[~ind], y_pred_ensemble2.flatten()[~ind], s=0.1, color='k')
plt.scatter(Y[ind], y_pred_ensemble2.flatten()[ind], s=0.5, color='r', label="STD_y > 0.05")
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (mean)")
plt.legend()

plt.sca(axarr[1])
plt.scatter(Y, y_pred_STD2.flatten(), s=0.1, color='k')
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (STD)")

plt.sca(axarr[2])
plt.scatter(training_data['mag_i_lsst'], y_pred_STD2.flatten(), s=0.1, color='k')
plt.xlabel("i magnitude")
plt.ylabel("predicted redshift (STD)")

plt.tight_layout()

## Cardinal Y10

In [ ]:
training_data = train_dataset_taskset1['cardinal'][10]
test_data = test_dataset_taskset1['cardinal'][10]

X, Y = cnnpz.transform_data_to_XY(training_data, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test, _ = cnnpz.transform_data_to_XY(test_data, lambda_array_cen, filter_blocks, apply_stretch = False, missingY=True)

In [ ]:
trained_models, histories = cnnpz.train_ensembles(build_model, X, Y)

In [ ]:
cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.1))

In [ ]:
y_pred_ensemble2, y_pred_STD2 = cnnpz.ensemble_predict(trained_models, X)

In [ ]:
# save the trained model:
root = "/pscratch/sd//jaimerz/cnnpz/PZDC/taskset1/Cardinal_Y10/models"
save_dir = root + "/ensemble_CNN_6layers"
cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
fname = save_dir + "/stats.pkl"
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y, y_pred_ensemble2.flatten(), training_data['mag_i_lsst'], save=True, saveroot=fname, 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y, y_pred_ensemble2.flatten(), 
           redshift_bins, imag_bins, training_data['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# also plot the STD as a function of redshift and i-band magnitude:
fig,axarr=plt.subplots(1,3,figsize=[13,4])
plt.sca(axarr[0])
ind = y_pred_STD2.flatten() >= 0.05
plt.scatter(Y[~ind], y_pred_ensemble2.flatten()[~ind], s=0.1, color='k')
plt.scatter(Y[ind], y_pred_ensemble2.flatten()[ind], s=0.5, color='r', label="STD_y > 0.05")
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (mean)")
plt.legend()

plt.sca(axarr[1])
plt.scatter(Y, y_pred_STD2.flatten(), s=0.1, color='k')
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (STD)")

plt.sca(axarr[2])
plt.scatter(training_data['mag_i_lsst'], y_pred_STD2.flatten(), s=0.1, color='k')
plt.xlabel("i magnitude")
plt.ylabel("predicted redshift (STD)")

plt.tight_layout()

## Flagship

In [ ]:
training_data = train_dataset_taskset1['flagship'][1]
test_data = test_dataset_taskset1['flagship'][1]

X, Y = cnnpz.transform_data_to_XY(training_data, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test, _ = cnnpz.transform_data_to_XY(test_data, lambda_array_cen, filter_blocks, apply_stretch = False, missingY=True)